In [ ]:
# Example 1: Loading and Exploring Datasets
try:
    from datasets import load_dataset, Dataset
    import pandas as pd
    import json
    import os
    import tempfile
    
    print("=== Loading a Small Dataset ===")
    _tmp = tempfile.mkdtemp()
    _data_file = os.path.join(_tmp, "reviews.jsonl")
    with open(_data_file, "w") as f:
        for i in range(150):
            f.write(json.dumps({"text": f"Review number {i}: " + ("a fine film" if i % 2 else "a dull film"), "label": i % 2}) + "\n")
    dataset = load_dataset("json", data_files=_data_file, split="train[:100]", cache_dir=os.path.join(_tmp, "cache"))
    assert len(dataset) == 100, len(dataset)
    assert dataset.column_names == ["text", "label"], dataset.column_names
    assert dataset[1]["label"] == 1 and dataset[1]["text"].startswith("Review number 1:"), dataset[1]
    
    print(f"Dataset type: {type(dataset)}")
    print(f"Dataset length: {len(dataset)}")
    print(f"Dataset features: {dataset.features}")
    print(f"Column names: {dataset.column_names}")
    
    # Examine first few examples
    print(f"\nFirst example text: {dataset[0]['text'][:100]}...")
    print(f"First example label: {dataset[0]['label']}")
    
    print("\n=== Creating Custom Dataset ===")
    # Create dataset from dictionary
    custom_data = {
        'text': ['This is great!', 'Not so good.', 'Amazing work!'],
        'label': [1, 0, 1],
        'id': [1, 2, 3]
    }
    custom_dataset = Dataset.from_dict(custom_data)
    print(f"Custom dataset: {custom_dataset}")
    print(f"Custom features: {custom_dataset.features}")
    
    print("Example 1: Loading and exploring datasets test passed")
    
except Exception as e:
    print(f"Example 1: Error in dataset loading test: {e}")
    import traceback
    traceback.print_exc()
    raise

In [ ]:
# Example 2: Data processing and transformation
try:
    from datasets import Dataset
    
    print("=== Creating Sample Dataset ===")
    # Create a sample text classification dataset
    sample_data = {
        'text': [
            'I love this product!',
            'This is terrible quality.',
            'Amazing customer service.',
            'Not worth the money.',
            'Highly recommended!'
        ],
        'label': [1, 0, 1, 0, 1],
        'length': [0, 0, 0, 0, 0]  # Will be calculated
    }
    dataset = Dataset.from_dict(sample_data)
    
    print("\n=== Text Processing Transformations ===")
    # Add text length feature
    def add_length(examples):
        examples['length'] = len(examples['text'])
        examples['word_count'] = len(examples['text'].split())
        return examples
    
    dataset = dataset.map(add_length)
    
    # Filter examples by length
    long_texts = dataset.filter(lambda x: x['length'] > 15)
    print(f"Long texts count: {len(long_texts)}")
    
    # Select subset of columns
    text_only = dataset.select_columns(['text', 'word_count'])
    print(f"Text-only dataset columns: {text_only.column_names}")
    
    # Sort by word count
    sorted_dataset = dataset.sort('word_count')
    print(f"Sorted by word count: {[ex['word_count'] for ex in sorted_dataset]}")
    
    print("Example 2: Data processing test passed")
    
except Exception as e:
    print(f"Example 2: Error in processing test: {e}")
    import traceback
    traceback.print_exc()
    raise

In [ ]:
# Example 3: Dataset splitting and batching
try:
    from datasets import Dataset, DatasetDict
    import random
    
    print("=== Creating Training Dataset ===")
    # Create a larger sample dataset
    texts = [f'Sample text {i} with content' for i in range(20)]
    labels = [random.randint(0, 1) for _ in range(20)]
    
    dataset = Dataset.from_dict({
        'text': texts,
        'label': labels,
        'id': list(range(20))
    })
    
    print(f"Original dataset size: {len(dataset)}")
    
    print("\n=== Train/Test Split ===")
    # Split dataset into train/test
    train_test = dataset.train_test_split(test_size=0.3, seed=42)
    
    print(f"Train size: {len(train_test['train'])}")
    print(f"Test size: {len(train_test['test'])}")
    
    # Create a DatasetDict
    dataset_dict = DatasetDict({
        'train': train_test['train'],
        'test': train_test['test']
    })
    print(f"Dataset dict keys: {list(dataset_dict.keys())}")
    
    print("\n=== Shuffling and Batching ===")
    # Shuffle dataset
    shuffled = dataset.shuffle(seed=42)
    print(f"Shuffled first 3 IDs: {[shuffled[i]['id'] for i in range(3)]}")
    
    # Create batches for processing
    def process_batch(batch):
        return {
            'processed_text': [f"PROC: {text}" for text in batch['text']],
            'batch_size': [len(batch['text'])] * len(batch['text'])
        }
    
    batched = dataset.map(process_batch, batched=True, batch_size=5)
    print(f"Batch processed example: {batched[0]['processed_text']}")
    
    print("Example 3: Dataset splitting and batching test passed")
    
except Exception as e:
    print(f"Example 3: Error in splitting test: {e}")
    import traceback
    traceback.print_exc()
    raise

In [ ]:
# Example 4: Dataset serialization and caching
try:
    from datasets import Dataset, load_from_disk
    import tempfile
    import os
    
    print("=== Creating Dataset for Serialization ===")
    # Create a dataset with various data types
    dataset = Dataset.from_dict({
        'text': ['Hello world', 'Python is great', 'Machine learning'],
        'numbers': [1.5, 2.7, 3.9],
        'categories': ['A', 'B', 'A'],
        'flags': [True, False, True]
    })
    
    print(f"Original dataset: {dataset}")
    print(f"Dataset info: {dataset.info}")
    
    print("\n=== Saving and Loading Dataset ===")
    # Create temporary directory for saving
    with tempfile.TemporaryDirectory() as temp_dir:
        dataset_path = os.path.join(temp_dir, 'test_dataset')
        
        # Save dataset
        dataset.save_to_disk(dataset_path)
        print(f"Dataset saved to: {dataset_path}")
        
        # Load dataset back
        loaded_dataset = load_from_disk(dataset_path)
        print(f"Loaded dataset: {loaded_dataset}")
        print(f"Data preserved: {dataset[0] == loaded_dataset[0]}")
    
    print("\n=== Dataset Features and Schema ===")
    # Examine dataset schema
    print(f"Features: {dataset.features}")
    print(f"Number of rows: {dataset.num_rows}")
    print(f"Number of columns: {dataset.num_columns}")
    
    # Convert to pandas for interoperability
    df = dataset.to_pandas()
    print(f"As pandas DataFrame shape: {df.shape}")
    print(f"DataFrame dtypes: {df.dtypes.to_dict()}")
    
    print("Example 4: Dataset serialization test passed")
    
except Exception as e:
    print(f"Example 4: Error in serialization test: {e}")
    import traceback
    traceback.print_exc()
    raise

In [ ]:
# Example 5: Advanced dataset operations and metrics
try:
    from datasets import Dataset, concatenate_datasets
    import numpy as np
    
    print("=== Creating Multiple Datasets ===")
    # Create first dataset
    dataset1 = Dataset.from_dict({
        'text': ['First dataset example', 'Another example'],
        'label': [1, 0],
        'source': ['dataset1', 'dataset1']
    })
    
    # Create second dataset
    dataset2 = Dataset.from_dict({
        'text': ['Second dataset example', 'Different text'],
        'label': [0, 1],
        'source': ['dataset2', 'dataset2']
    })
    
    print(f"Dataset 1 size: {len(dataset1)}")
    print(f"Dataset 2 size: {len(dataset2)}")
    
    print("\n=== Dataset Concatenation ===")
    # Concatenate datasets
    combined = concatenate_datasets([dataset1, dataset2])
    print(f"Combined dataset size: {len(combined)}")
    print(f"Sources: {combined['source']}")
    
    print("\n=== Advanced Transformations ===")
    # Complex mapping with indices
    def add_metadata(example, idx):
        example['index'] = idx
        example['text_length'] = len(example['text'])
        example['is_long'] = len(example['text']) > 15
        return example
    
    enriched = combined.map(add_metadata, with_indices=True)
    
    # Group by label and compute statistics
    label_groups = enriched.filter(lambda x: x['label'] == 1)
    print(f"Positive examples: {len(label_groups)}")
    
    avg_length = np.mean([ex['text_length'] for ex in enriched])
    print(f"Average text length: {avg_length:.2f}")
    
    print("\n=== Dataset Performance Info ===")
    # Dataset format and backend info
    print(f"Dataset format: {enriched.format}")
    print(f"Cache files: {len(enriched.cache_files) if enriched.cache_files else 0}")
    
    print("Example 5: Advanced operations test passed")
    
except Exception as e:
    print(f"Example 5: Error in advanced operations test: {e}")
    import traceback
    traceback.print_exc()
    raise